# Options Edge Scanner: Public Demo

This 2–3 minute, fully offline walkthrough shows how the project turns a local option-quote universe into an inspectable **research shortlist** of debit vertical spreads. The output is a ranked research artifact—not a trade recommendation, performance claim, or brokerage instruction.

> **Demo data:** the committed `XYZ` test fixture is deliberately synthetic. It is small, deterministic, and provenance-bearing so the real production pipeline can be shown safely without network access, credentials, or licensed market data.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import display

from options_edge_scanner.config import ScannerConfig
from options_edge_scanner.data.loader import load_option_quotes
from options_edge_scanner.scanner.engine import run_scan

REPO_ROOT = Path.cwd()
INPUT_PATH = REPO_ROOT / 'tests' / 'fixtures' / 'sample_option_chain.csv'
AS_OF = '2026-01-02'

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        'Launch Jupyter from the repository root so the committed demo fixture is available.'
    )

print('Offline demo input:', INPUT_PATH.relative_to(REPO_ROOT))
print('Dataset: synthetic, committed test fixture | Network access: not required')

## 1. Load a small, traceable option universe

The supported loader validates the canonical local-file shape, normalizes dates/types, and supplies a midpoint from bid/ask when needed. This fixture carries quote and spot timestamps, so the scanner can enforce its point-in-time checks.

In [ ]:
option_quotes = load_option_quotes(INPUT_PATH)

universe_columns = [
    'ticker', 'expiration', 'type', 'strike', 'bid', 'ask', 'mid',
    'implied_vol', 'underlying_price', 'quote_observed_at',
    'underlying_price_observed_at', 'underlying_price_source',
]
universe_view = option_quotes.loc[:, universe_columns].copy()
universe_view['expiration'] = universe_view['expiration'].dt.strftime('%Y-%m-%d')
universe_view['quote_observed_at'] = universe_view['quote_observed_at'].dt.strftime('%Y-%m-%d %H:%M')
universe_view['underlying_price_observed_at'] = (
    universe_view['underlying_price_observed_at'].dt.strftime('%Y-%m-%d %H:%M')
)

display(universe_view.round({'bid': 2, 'ask': 2, 'mid': 2, 'implied_vol': 3, 'underlying_price': 2}))
print(f'{len(option_quotes)} quotes loaded for one synthetic ticker and expiration.')

## 2. Run the real scanner and capture its audit report

This calls the maintained config-driven `run_scan(...)` API—there is no notebook scoring logic. The API loads and enriches the quotes, verifies provenance, applies quote and spread quality checks, generates debit verticals, applies execution adjustments, calculates probability/EV fields, and ranks the survivors. The production CSV and JSON report are written to a temporary local directory for this demo.

In [ ]:
with TemporaryDirectory() as artifact_directory:
    artifact_directory = Path(artifact_directory)
    opportunities = run_scan(
        ScannerConfig(
            input_path=INPUT_PATH,
            output_path=artifact_directory / 'ranked_opportunities.csv',
            report_output_path=artifact_directory / 'scan_report.json',
            as_of=AS_OF,
        )
    )
    scan_report = json.loads((artifact_directory / 'scan_report.json').read_text(encoding='utf-8'))

counts = scan_report['counts']
pipeline_view = pd.DataFrame(
    [
        ('Validated and provenance-checked quotes', counts['prepared_quote_count']),
        ('Quotes surviving quality filters', counts['filtered_quote_count']),
        ('Vertical spreads generated', counts['generated_spread_count']),
        ('Spreads surviving quality and execution checks', counts['filtered_spread_count']),
        ('Spreads with positive days to expiration', counts['dte_eligible_spread_count']),
        ('Ranked shortlist', counts['ranked_candidate_count']),
    ],
    columns=['pipeline stage', 'rows'],
)

display(pipeline_view)
model = scan_report['probability_models'][0]
print(f"Probability model: {model['probability_model_name']} v{model['probability_model_version']}")
print('Report warnings:', ', '.join(scan_report['warnings']) if scan_report['warnings'] else 'none')

## 3. Inspect the ranked research output

The table keeps the decision-relevant structure: contract geometry, quoted versus execution-adjusted entry cost, modeled probability and EV, liquidity, and the scanner-generated rank. Values are rounded only for display.

In [ ]:
result_columns = [
    'selection_rank', 'spread_type', 'lower_strike', 'upper_strike', 'days_to_expiration',
    'quoted_debit', 'debit', 'probability_of_profit', 'expected_value',
    'liquidity_score', 'ranking_score',
]
ranked_view = opportunities.loc[:, result_columns].rename(
    columns={
        'selection_rank': 'rank',
        'spread_type': 'strategy',
        'lower_strike': 'lower strike',
        'upper_strike': 'upper strike',
        'days_to_expiration': 'DTE',
        'quoted_debit': 'quoted debit',
        'debit': 'entry debit',
        'probability_of_profit': 'model probability',
        'expected_value': 'execution-adjusted EV',
        'liquidity_score': 'liquidity',
        'ranking_score': 'ranking score',
    }
)

display(ranked_view.round(3))

## 4. Explain the top-ranked candidate with its own evidence

Rank 1 has the largest scanner-generated ranking score in this fixture. Its entry debit is higher than the quoted midpoint because the real pipeline applies a conservative execution adjustment; its ranking also retains a liquidity diagnostic. Those are research inputs and assumptions, not evidence that the spread will be profitable.

In [ ]:
top = opportunities.iloc[0]
top_evidence = pd.DataFrame(
    [
        ('contract', f"{top['spread_type']} | {top['lower_strike']:.0f}/{top['upper_strike']:.0f} | {top['days_to_expiration']:.0f} DTE"),
        ('selection rank / ranking score', f"{int(top['selection_rank'])} / {top['ranking_score']:.3f}"),
        ('quoted vs. execution-adjusted debit', f"{top['quoted_debit']:.2f} -> {top['debit']:.2f}"),
        ('entry slippage', f"{top['entry_slippage']:.2f} ({top['entry_slippage_pct']:.1%})"),
        ('model probability / execution-adjusted EV', f"{top['probability_of_profit']:.1%} / {top['expected_value']:.2f}"),
        ('liquidity score', f"{top['liquidity_score']:.3f}"),
        ('probability model', f"{top['probability_model_name']} v{top['probability_model_version']}"),
        ('predicted event', f"{top['predicted_event_direction']} terminal underlying {top['predicted_event_operator']} {top['predicted_event_threshold']:.2f} at {top['predicted_event_horizon']}"),
        ('quote / spot provenance', f"{top['quote_observed_at']:%Y-%m-%d %H:%M} | {top['underlying_price_source']}"),
    ],
    columns=['evidence field', 'reported value'],
)

display(top_evidence)

## 5. Close with auditability and evaluation boundaries

The report below is derived from the scanner's real JSON audit artifact. It records stage counts, model identity, run date, and warnings; the ranked rows retain the event semantics and provenance needed for later evaluation. This synthetic demo fixture has **no realized outcomes**, and the repository does not currently have enough accessible historical coverage to establish predictive effectiveness. Missing outcomes are not treated as wins, losses, or zeros.

In [ ]:
audit_view = pd.DataFrame(
    [
        ('local input', INPUT_PATH.relative_to(REPO_ROOT).as_posix()),
        ('scan as-of', scan_report['as_of']),
        ('quoted / ranked rows', f"{counts['prepared_quote_count']} / {counts['ranked_candidate_count']}"),
        ('probability model', f"{model['probability_model_name']} v{model['probability_model_version']}"),
        ('report warnings', ', '.join(scan_report['warnings']) if scan_report['warnings'] else 'none'),
        ('realized-outcome coverage', 'Not included in this synthetic fixture; effectiveness remains unproven.'),
    ],
    columns=['audit item', 'value'],
)

display(audit_view)

### Takeaway

> The scanner turns a broad options universe into a structured, inspectable shortlist while preserving the evidence, execution assumptions, exclusions, and data limitations behind each ranking.